# Scam Message Detection with Embeddings

I keep seeing scam messages that use similar ideas — fake prizes, fake company messages, and “earn money daily” offers.

I want to test whether embeddings can help compare a new message with known scam examples.

My question is: if a message looks semantically similar to known scams, is that enough to warn someone?

## The data

I'm starting with a small set of scam messages and legitimate messages to compare.

In [1]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Known scam messages
scam_messages = [
    "Congratulations! You won a cash prize. Send your bank details to claim it.",
    "Your bank account will be suspended. Verify your information immediately.",
    "Earn money every day from home. No experience needed.",
    "We are contacting you from the company. Confirm your personal information to continue."
]

# Legitimate messages for comparison
legitimate_messages = [
    "Your package is scheduled for delivery tomorrow.",
    "Your monthly bank statement is now available in the official app.",
    "The meeting has been moved to 3 PM.",
    "Your order has been shipped and will arrive this week."
]

## Step 1: Convert the messages into embeddings

In [5]:
# Convert both sets into embeddings
scam_embeddings = model.encode(scam_messages)
legit_embeddings = model.encode(legitimate_messages)

print("Scam embeddings shape:", scam_embeddings.shape)
print("Legitimate embeddings shape:", legit_embeddings.shape)

Scam embeddings shape: (4, 384)
Legitimate embeddings shape: (4, 384)


## Step 2: Test a new message

Now I want to take a message the model has never seen before, and check 
if it's closer to the scam examples or the legitimate ones.

In [6]:
# A new message that just arrived
new_message = "You have been selected to receive a daily income. Contact us now to start."

new_embedding = model.encode([new_message])

# Compare it with both groups
scam_similarity = cos_sim(new_embedding, scam_embeddings)
legit_similarity = cos_sim(new_embedding, legit_embeddings)

print("Similarity with scam messages:", scam_similarity)
print("Similarity with legitimate messages:", legit_similarity)

Similarity with scam messages: tensor([[0.4066, 0.3788, 0.4145, 0.5439]])
Similarity with legitimate messages: tensor([[0.3422, 0.4172, 0.1850, 0.3404]])


In [8]:
test_message = "Your bank statement is ready. Log in through the official app."

test_embedding = model.encode([test_message])

scam_similarity = cos_sim(test_embedding, scam_embeddings)
legit_similarity = cos_sim(test_embedding, legit_embeddings)

print("Similarity with scam messages:", scam_similarity)
print("Similarity with legitimate messages:", legit_similarity)

Similarity with scam messages: tensor([[0.4345, 0.5538, 0.1151, 0.5032]])
Similarity with legitimate messages: tensor([[0.2899, 0.7934, 0.1664, 0.3059]])


### Testing how much the reference examples matter

The previous legitimate message matched strongly with a very similar legitimate example.

I want to remove that close example and run the same test again.

The message and the model stay the same. Only the reference examples change.

In [9]:
# Remove the most similar legitimate example for this experiment
legitimate_without_bank = [
    legitimate_messages[0],
    legitimate_messages[2],
    legitimate_messages[3]
]

legitimate_without_bank_embeddings = model.encode(legitimate_without_bank)

scam_similarity = cos_sim(test_embedding, scam_embeddings)
legitimate_similarity = cos_sim(
    test_embedding,
    legitimate_without_bank_embeddings
)

print("Similarity with scam messages:", scam_similarity)
print("Similarity with legitimate messages:", legitimate_similarity)

print("\nBest scam match:", scam_similarity.max().item())
print("Best legitimate match:", legitimate_similarity.max().item())

Similarity with scam messages: tensor([[0.4345, 0.5538, 0.1151, 0.5032]])
Similarity with legitimate messages: tensor([[0.2899, 0.1664, 0.3059]])

Best scam match: 0.553814172744751
Best legitimate match: 0.30585208535194397


### Result

Interesting — the result changed.

Before removing the example:
- Scam: 0.55
- Legitimate: 0.79

After removing it:
- Scam: 0.55
- Legitimate: 0.31

The message didn't change. I only changed the examples I was comparing it with.

So can I really depend on similarity alone?

### A less obvious example

What happens if the scam message doesn't use obvious words like "prize" or "bank account"?


In [11]:
test_message = """
Hi, we are looking for people to help rate products online.
It only takes a few minutes and you will receive a payment after each task.
"""

test_embedding = model.encode([test_message])

scam_similarity = cos_sim(test_embedding, scam_embeddings)
legitimate_similarity = cos_sim(test_embedding, legit_embeddings)
print("Similarity with scam messages:", scam_similarity)
print("Similarity with legitimate messages:", legitimate_similarity)

print("\nBest scam match:", scam_similarity.max().item())
print("Best legitimate match:", legitimate_similarity.max().item())

Similarity with scam messages: tensor([[0.3287, 0.1963, 0.3355, 0.3915]])
Similarity with legitimate messages: tensor([[0.2524, 0.2379, 0.1071, 0.2796]])

Best scam match: 0.3915242552757263
Best legitimate match: 0.27955594658851624


### Result

It was still closer to the scam examples, even without obvious scam words.

But the best match was only 0.39, so similarity alone still doesn't feel like enough to make a decision.

### Moving to real data

The 8 messages were useful for testing the idea, but they're not enough to train a classifier.

I want to try this with real messages next. I'm also interested in Arabic scam messages, especially the kind I keep seeing on WhatsApp and Telegram.

I couldn't find a dataset that really matches that yet, so I might build a small one myself.